# Best-Estimate Plaque Type Table by Artery

This Colab notebook estimates plaque composition per artery from OpenPlaque nnU-Net masks and vessel-wide HU candidate regions. It is designed to produce a concise table for LAD, RCA, and LCX.

Research use only. Not clinically validated. Not for diagnosis.

## 1. Mount Google Drive


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 2. Install and clone/pull OpenPlaque


In [ ]:
from pathlib import Path
import inspect, os, sys, subprocess

REPO_URL = 'https://github.com/pazzani/OpenPlaque.git'
REPO_DIR = Path('/content/OpenPlaque')
REPO_BRANCH = 'agent/plaque-context-verification'

if REPO_DIR.exists():
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', 'origin', REPO_BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', '-B', REPO_BRANCH, f'origin/{REPO_BRANCH}'], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'reset', '--hard', f'origin/{REPO_BRANCH}'], check=True)
else:
    subprocess.run(['git', 'clone', '--branch', REPO_BRANCH, REPO_URL, str(REPO_DIR)], check=True)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(REPO_DIR / 'requirements-colab.txt')], check=False)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--force-reinstall', '--no-deps', '-e', str(REPO_DIR)], check=True)
sys.path.insert(0, str(REPO_DIR / 'src'))

import openplaque
import openplaque.plaque_context as plaque_context
commit = subprocess.check_output(['git', '-C', str(REPO_DIR), 'rev-parse', '--short', 'HEAD'], text=True).strip()
signature = inspect.signature(plaque_context.vessel_wide_candidate_mask)
print('Using', REPO_DIR)
print('Commit', commit)
print('openplaque from', openplaque.__file__)
print('vessel_wide_candidate_mask signature', signature)


## 3. Configure inputs and estimator thresholds


In [ ]:
DRIVE_ROOT = Path('/content/drive/MyDrive/OpenPlaque')
STUDY_ZIP = DRIVE_ROOT / 'Full_DICOM.zip'
MODEL_ZIP = DRIVE_ROOT / 'models' / 'Dataset001_CCTA_DHM-20260703T233210Z-3-001.zip'
OUTPUT_DIR = DRIVE_ROOT / 'UCLA_Plaque_Type_Estimates'
MASK_DIR = OUTPUT_DIR / 'nnunet_masks'
PLOT_DIR = OUTPUT_DIR / 'plots'
MASK_SEARCH_DIRS = [
    OUTPUT_DIR / 'nnunet_masks',
    DRIVE_ROOT / 'UCLA_Plaque_Context_Verification' / 'nnunet_masks',
]
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MASK_DIR.mkdir(parents=True, exist_ok=True)
PLOT_DIR.mkdir(parents=True, exist_ok=True)

FALLBACK_SERIES = {'RCA': 1035, 'LCX': 1039, 'LAD': 1043}
VESSELS = ['LAD', 'RCA', 'LCX']
REUSE_SEGMENTATIONS = True
RUN_NNUNET_IF_MISSING = True

# The artery DICOM/nnU-Net volumes can be stored as 1 mm isotropic derived images.
# For physical plaque volume estimates on the UCLA study, use the original CTA
# acquisition spacing from Series 7 when artery-series spacing is implausibly large.
USE_REFERENCE_SERIES_SPACING_WHEN_ARTERY_SPACING_IS_1MM = True
REFERENCE_SPACING_SERIES_NUMBER = 7
MANUAL_REFERENCE_SPACING_XYZ_MM = (0.3515625, 0.3515625, 0.3)
MAX_EXPECTED_CCTA_VOXEL_VOLUME_MM3 = 0.5

VESSEL_DILATION_VOXELS = 1
EXCLUDE_VESSEL_LABEL_FOR_CANDIDATES = True
LOW_ATTENUATION_MIN_HU = -30
LOW_ATTENUATION_MAX_HU = 30
NONCALCIFIED_MIN_HU = 30
NONCALCIFIED_MAX_HU = 130
MIXED_MIN_HU = 130
MIXED_MAX_HU = 350
CALCIFIED_MIN_HU = 350

print('Study:', STUDY_ZIP)
print('Outputs:', OUTPUT_DIR)


## 4. Install or expose nnU-Net model


In [ ]:
import os, zipfile

NNUNET_RESULTS = Path('/content/nnUNet_results')
os.environ['nnUNet_results'] = str(NNUNET_RESULTS)
os.environ.setdefault('nnUNet_raw', '/content/nnUNet_raw')
os.environ.setdefault('nnUNet_preprocessed', '/content/nnUNet_preprocessed')

if MODEL_ZIP.exists() and not (NNUNET_RESULTS / 'Dataset001_CCTA_DHM').exists():
    NNUNET_RESULTS.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(MODEL_ZIP) as zf:
        zf.extractall(NNUNET_RESULTS)

print('nnUNet_results =', os.environ['nnUNet_results'])
print('Model zip exists:', MODEL_ZIP.exists(), MODEL_ZIP)


## 5. Load study and run or reuse segmentations


In [ ]:
import numpy as np
import pandas as pd
import SimpleITK as sitk
import os
import pydicom
from openplaque.study import OpenPlaqueStudy
from openplaque.run_new_data import auto_detect_or_fallback_series
from openplaque.segmentation import SegmentationReport, export_for_nnunet, run_nnunet, load_segmentation

study = OpenPlaqueStudy(str(STUDY_ZIP))
series_map = auto_detect_or_fallback_series(study, fallback_series=FALLBACK_SERIES)
series_map = {k: int(series_map[k]) for k in VESSELS}
print('Series map:', series_map)

def spacing_from_dicom_files(files, fallback_spacing):
    if not files:
        return tuple(float(x) for x in fallback_spacing), 'sitk_fallback_no_files'
    datasets = []
    for path in files:
        try:
            ds = pydicom.dcmread(path, stop_before_pixels=True, force=True)
            datasets.append(ds)
        except Exception:
            continue
    if not datasets:
        return tuple(float(x) for x in fallback_spacing), 'sitk_fallback_no_readable_dicom'

    first = datasets[0]
    try:
        row_spacing, col_spacing = [float(x) for x in first.PixelSpacing]
    except Exception:
        row_spacing, col_spacing = float(fallback_spacing[1]), float(fallback_spacing[0])

    positions = []
    for ds in datasets:
        ipp = getattr(ds, 'ImagePositionPatient', None)
        if ipp is not None and len(ipp) == 3:
            positions.append(np.asarray([float(x) for x in ipp], dtype=float))
    z_spacing = None
    if len(positions) > 1:
        positions = sorted(positions, key=lambda p: tuple(p.tolist()))
        deltas = [float(np.linalg.norm(positions[i + 1] - positions[i])) for i in range(len(positions) - 1)]
        deltas = [d for d in deltas if d > 1e-6]
        if deltas:
            z_spacing = float(np.median(deltas))
    if z_spacing is None:
        for attr in ('SpacingBetweenSlices', 'SliceThickness'):
            value = getattr(first, attr, None)
            if value is not None:
                try:
                    z_spacing = float(value)
                    break
                except Exception:
                    pass
    if z_spacing is None:
        z_spacing = float(fallback_spacing[2])

    return (float(col_spacing), float(row_spacing), float(z_spacing)), 'dicom_metadata'

def dicom_files_for_series_number(study, series_number):
    matches = [s for s in study.series if int(s.get('series_number', -1)) == int(series_number)]
    if not matches:
        return []
    folder = matches[0]['folder']
    files = []
    for root, _, names in os.walk(folder):
        for name in names:
            files.append(os.path.join(root, name))
    return files

def reference_spacing_for_volume(study):
    files = dicom_files_for_series_number(study, REFERENCE_SPACING_SERIES_NUMBER)
    if files:
        spacing, source = spacing_from_dicom_files(files, MANUAL_REFERENCE_SPACING_XYZ_MM)
        if float(np.prod(spacing)) < MAX_EXPECTED_CCTA_VOXEL_VOLUME_MM3:
            return spacing, f'reference_series_{REFERENCE_SPACING_SERIES_NUMBER}_{source}'
    return tuple(float(x) for x in MANUAL_REFERENCE_SPACING_XYZ_MM), 'manual_reference_spacing'

REFERENCE_VOLUME_SPACING_XYZ, REFERENCE_VOLUME_SPACING_SOURCE = reference_spacing_for_volume(study)
print('Reference volume spacing:', REFERENCE_VOLUME_SPACING_XYZ, REFERENCE_VOLUME_SPACING_SOURCE)

DICOM_SPACING_BY_VESSEL = {}
DICOM_SPACING_SOURCE_BY_VESSEL = {}
reports = []
for vessel in VESSELS:
    image, volume, dicom_files = study.load_series(series_map[vessel])
    dicom_spacing, spacing_source = spacing_from_dicom_files(dicom_files, image.GetSpacing())
    volume_spacing = dicom_spacing
    volume_spacing_source = spacing_source
    if (
        USE_REFERENCE_SERIES_SPACING_WHEN_ARTERY_SPACING_IS_1MM
        and float(np.prod(dicom_spacing)) >= MAX_EXPECTED_CCTA_VOXEL_VOLUME_MM3
    ):
        volume_spacing = REFERENCE_VOLUME_SPACING_XYZ
        volume_spacing_source = REFERENCE_VOLUME_SPACING_SOURCE
    DICOM_SPACING_BY_VESSEL[vessel] = volume_spacing
    DICOM_SPACING_SOURCE_BY_VESSEL[vessel] = volume_spacing_source
    print(
        f'{vessel} SimpleITK spacing: {tuple(float(x) for x in image.GetSpacing())}; '
        f'artery DICOM spacing: {dicom_spacing} ({spacing_source}); '
        f'volume spacing used: {volume_spacing} ({volume_spacing_source})'
    )
    saved_mask = next((d / f'{vessel}.nii.gz' for d in MASK_SEARCH_DIRS if (d / f'{vessel}.nii.gz').exists()), None)
    if REUSE_SEGMENTATIONS and saved_mask is not None:
        mask_image = sitk.ReadImage(str(saved_mask))
        mask = sitk.GetArrayFromImage(mask_image)
        input_dir = str(OUTPUT_DIR / f'{vessel}_input')
        output_dir = str(saved_mask.parent)
        print(f'Reused {saved_mask}')
    elif RUN_NNUNET_IF_MISSING:
        MASK_DIR.mkdir(parents=True, exist_ok=True)
        input_dir = str(OUTPUT_DIR / f'{vessel}_input')
        output_dir = str(OUTPUT_DIR / f'{vessel}_nnunet_output')
        export_for_nnunet(image, input_dir, vessel)
        try:
            run_nnunet(input_dir, output_dir)
        except subprocess.CalledProcessError as exc:
            model_dir = NNUNET_RESULTS / 'Dataset001_CCTA_DHM'
            raise RuntimeError(
                f'nnU-Net failed for {vessel}. Check that the model exists at {model_dir}, '
                f'that MODEL_ZIP points to the uploaded model zip, and that the Colab runtime has GPU enabled. '
                f'Original command: {exc.cmd}'
            ) from exc
        mask_image, mask = load_segmentation(output_dir, vessel)
        saved_mask = MASK_DIR / f'{vessel}.nii.gz'
        sitk.WriteImage(mask_image, str(saved_mask))
        print(f'Ran nnU-Net and saved {saved_mask}')
    else:
        searched = ', '.join(str(d / f'{vessel}.nii.gz') for d in MASK_SEARCH_DIRS)
        raise FileNotFoundError(f'Missing saved mask for {vessel}. Searched: {searched}')
    reports.append(SegmentationReport(vessel, image, mask_image, volume, mask, input_dir, output_dir))

[(r.name, r.label_counts(), r.hu_statistics()) for r in reports]


## 6. Estimate plaque types by artery

Best-estimate method: classify all nnU-Net plaque-core voxels by HU, then add vessel-context candidate voxels outside labels 1 and 2 for low attenuation, noncalcified, and mixed/intermediate plaque. The table keeps strict nnU-Net core volume separate from context-expanded candidate volume.

Reference staging columns compare this research proxy with published patient-level TPV/NCPV thresholds from PROMISE and CONFIRM2. They are included for orientation only; those papers used validated quantitative CCTA tools with lumen and outer-vessel-wall contours.

Volumes use artery DICOM spacing when plausible. If the artery series is a 1 mm isotropic derived image, the notebook falls back to UCLA Series 7/reference CTA spacing for physical volume estimates. This avoids treating voxel counts as mm3.


In [ ]:
import importlib
import numpy as np
import pandas as pd
import openplaque.plaque_context as plaque_context
plaque_context = importlib.reload(plaque_context)
vessel_context_mask = plaque_context.vessel_context_mask

PROMISE_TPV_THRESHOLD_MM3 = 87.2
CONFIRM2_TPV_STAGES_MM3 = (250.0, 750.0)
CONFIRM2_NCPV_STAGES_MM3 = (20.0, 125.0, 375.0)

def _count_range(values, lo=None, hi=None):
    values = np.asarray(values, dtype=float)
    cond = np.ones(values.shape, dtype=bool)
    if lo is not None:
        cond &= values >= float(lo)
    if hi is not None:
        cond &= values < float(hi)
    return int(cond.sum())

def _promise_flag(volume_mm3):
    return f'>= {PROMISE_TPV_THRESHOLD_MM3:g} mm3' if volume_mm3 >= PROMISE_TPV_THRESHOLD_MM3 else f'< {PROMISE_TPV_THRESHOLD_MM3:g} mm3'

def _confirm2_tpv_stage(volume_mm3):
    if volume_mm3 <= 0:
        return '0 mm3'
    if volume_mm3 <= CONFIRM2_TPV_STAGES_MM3[0]:
        return '>0-250 mm3'
    if volume_mm3 <= CONFIRM2_TPV_STAGES_MM3[1]:
        return '>250-750 mm3'
    return '>750 mm3'

def _confirm2_ncpv_stage(volume_mm3):
    if volume_mm3 < CONFIRM2_NCPV_STAGES_MM3[0]:
        return '0-20 mm3'
    if volume_mm3 <= CONFIRM2_NCPV_STAGES_MM3[1]:
        return '>20-125 mm3'
    if volume_mm3 <= CONFIRM2_NCPV_STAGES_MM3[2]:
        return '>125-375 mm3'
    return '>375 mm3'

def estimate_plaque_types(report):
    volume = np.asarray(report.volume)
    mask = np.asarray(report.mask)
    image_spacing_xyz = tuple(float(x) for x in DICOM_SPACING_BY_VESSEL.get(report.name, report.image.GetSpacing()))
    image_spacing_source = DICOM_SPACING_SOURCE_BY_VESSEL.get(report.name, 'image_spacing_fallback')
    sitk_image_spacing_xyz = tuple(float(x) for x in report.image.GetSpacing())
    mask_spacing_xyz = tuple(float(x) for x in report.mask_image.GetSpacing())
    voxel_volume = float(np.prod(image_spacing_xyz))
    core = mask == 2
    anatomy_band = vessel_context_mask(mask, vessel_label=1, plaque_label=2, vessel_dilation_voxels=VESSEL_DILATION_VOXELS, connectivity=26)
    candidate_region = anatomy_band & (mask != 2)
    if EXCLUDE_VESSEL_LABEL_FOR_CANDIDATES:
        candidate_region &= mask != 1

    core_vals = volume[core]
    cand_vals = volume[candidate_region]

    core_low = _count_range(core_vals, LOW_ATTENUATION_MIN_HU, LOW_ATTENUATION_MAX_HU)
    core_noncalc = _count_range(core_vals, NONCALCIFIED_MIN_HU, NONCALCIFIED_MAX_HU)
    core_mixed = _count_range(core_vals, MIXED_MIN_HU, MIXED_MAX_HU)
    core_calc = _count_range(core_vals, CALCIFIED_MIN_HU, None)

    cand_low = _count_range(cand_vals, LOW_ATTENUATION_MIN_HU, LOW_ATTENUATION_MAX_HU)
    cand_noncalc = _count_range(cand_vals, NONCALCIFIED_MIN_HU, NONCALCIFIED_MAX_HU)
    cand_mixed = _count_range(cand_vals, MIXED_MIN_HU, MIXED_MAX_HU)

    low = core_low + cand_low
    noncalc = core_noncalc + cand_noncalc
    mixed = core_mixed + cand_mixed
    calc = core_calc
    context_candidate = cand_low + cand_noncalc + cand_mixed
    total_proxy = low + noncalc + mixed + calc
    ncpv_proxy = noncalc + mixed
    strict_core = int(core.sum())

    def vol(vox):
        return float(vox * voxel_volume)

    row = {
        'artery': report.name,
        'image_spacing_source': image_spacing_source,
        'image_spacing_x_mm': image_spacing_xyz[0],
        'image_spacing_y_mm': image_spacing_xyz[1],
        'image_spacing_z_mm': image_spacing_xyz[2],
        'sitk_image_spacing_x_mm': sitk_image_spacing_xyz[0],
        'sitk_image_spacing_y_mm': sitk_image_spacing_xyz[1],
        'sitk_image_spacing_z_mm': sitk_image_spacing_xyz[2],
        'mask_spacing_x_mm': mask_spacing_xyz[0],
        'mask_spacing_y_mm': mask_spacing_xyz[1],
        'mask_spacing_z_mm': mask_spacing_xyz[2],
        'voxel_volume_mm3': voxel_volume,
        'low_attenuation_voxels': low,
        'low_attenuation_mm3': vol(low),
        'noncalcified_voxels': noncalc,
        'noncalcified_mm3': vol(noncalc),
        'mixed_intermediate_voxels': mixed,
        'mixed_intermediate_mm3': vol(mixed),
        'calcified_voxels': calc,
        'calcified_mm3': vol(calc),
        'strict_nnunet_core_voxels': strict_core,
        'strict_nnunet_core_volume_mm3': vol(strict_core),
        'calcified_core_voxels': core_calc,
        'calcified_core_volume_mm3': vol(core_calc),
        'context_selected_candidate_voxels': context_candidate,
        'context_expanded_candidate_volume_mm3': vol(context_candidate),
        'total_plaque_volume_proxy_voxels': total_proxy,
        'total_plaque_volume_proxy_mm3': vol(total_proxy),
        'confirm2_ncpv_proxy_voxels': ncpv_proxy,
        'confirm2_ncpv_proxy_mm3': vol(ncpv_proxy),
        'core_plaque_voxels': strict_core,
        'candidate_region_voxels': int(candidate_region.sum()),
        'candidate_low_attenuation_voxels': cand_low,
        'candidate_noncalcified_voxels': cand_noncalc,
        'candidate_mixed_intermediate_voxels': cand_mixed,
    }
    for key in ['low_attenuation', 'noncalcified', 'mixed_intermediate', 'calcified']:
        row[f'{key}_fraction'] = (row[f'{key}_voxels'] / total_proxy) if total_proxy else 0.0
    row['promise_tpv_proxy_flag'] = _promise_flag(row['total_plaque_volume_proxy_mm3'])
    row['confirm2_tpv_proxy_stage'] = _confirm2_tpv_stage(row['total_plaque_volume_proxy_mm3'])
    row['confirm2_ncpv_proxy_stage'] = _confirm2_ncpv_stage(row['confirm2_ncpv_proxy_mm3'])
    return row

rows = [estimate_plaque_types(r) for r in reports]
total_row = {'artery': 'TOTAL'}
sum_cols = [c for c in rows[0].keys() if c.endswith('_voxels') or c.endswith('_mm3')]
for col in sum_cols:
    total_row[col] = sum(r.get(col, 0) for r in rows)
total_proxy = total_row.get('total_plaque_volume_proxy_voxels', 0)
for key in ['low_attenuation', 'noncalcified', 'mixed_intermediate', 'calcified']:
    total_row[f'{key}_fraction'] = (total_row.get(f'{key}_voxels', 0) / total_proxy) if total_proxy else 0.0
for col in ['image_spacing_source', 'image_spacing_x_mm', 'image_spacing_y_mm', 'image_spacing_z_mm', 'sitk_image_spacing_x_mm', 'sitk_image_spacing_y_mm', 'sitk_image_spacing_z_mm', 'mask_spacing_x_mm', 'mask_spacing_y_mm', 'mask_spacing_z_mm', 'voxel_volume_mm3']:
    total_row[col] = ''
total_row['promise_tpv_proxy_flag'] = _promise_flag(total_row['total_plaque_volume_proxy_mm3'])
total_row['confirm2_tpv_proxy_stage'] = _confirm2_tpv_stage(total_row['total_plaque_volume_proxy_mm3'])
total_row['confirm2_ncpv_proxy_stage'] = _confirm2_ncpv_stage(total_row['confirm2_ncpv_proxy_mm3'])

estimate_df = pd.DataFrame(rows + [total_row])
csv_path = OUTPUT_DIR / 'best_estimate_plaque_types_by_artery.csv'
estimate_df.to_csv(csv_path, index=False)
display_cols = [
    'artery',
    'strict_nnunet_core_volume_mm3',
    'voxel_volume_mm3',
    'image_spacing_source',
    'context_expanded_candidate_volume_mm3',
    'total_plaque_volume_proxy_mm3',
    'confirm2_ncpv_proxy_mm3',
    'low_attenuation_mm3',
    'noncalcified_mm3',
    'mixed_intermediate_mm3',
    'calcified_mm3',
    'low_attenuation_fraction',
    'noncalcified_fraction',
    'mixed_intermediate_fraction',
    'calcified_fraction',
    'promise_tpv_proxy_flag',
    'confirm2_tpv_proxy_stage',
    'confirm2_ncpv_proxy_stage',
]
estimate_df[display_cols]


## 7. Plot and export report


In [ ]:
import matplotlib.pyplot as plt

required_cols = {'voxel_volume_mm3', 'total_plaque_volume_proxy_mm3', 'strict_nnunet_core_volume_mm3'}
missing_cols = required_cols.difference(estimate_df.columns)
if missing_cols:
    raise RuntimeError(f'estimate_df is stale; rerun section 6 after pulling the updated notebook. Missing columns: {sorted(missing_cols)}')

non_total = estimate_df[estimate_df['artery'] != 'TOTAL'].copy()
voxel_volumes = pd.to_numeric(non_total['voxel_volume_mm3'], errors='coerce')
if voxel_volumes.notna().any() and float(voxel_volumes.median()) >= 0.5:
    raise RuntimeError(
        'Voxel volume is unexpectedly large. Rerun sections 3, 5, and 6 so the UCLA Series 7/reference spacing fallback is applied.'
    )

plot_df = non_total.set_index('artery')
volume_cols = ['low_attenuation_mm3', 'noncalcified_mm3', 'mixed_intermediate_mm3', 'calcified_mm3']
colors = ['#d73027', '#fdae61', '#2b83ba', '#f7f7f7']
ax = plot_df[volume_cols].plot(kind='bar', stacked=True, figsize=(9, 5), color=colors, edgecolor='black')
ax.set_ylabel('Estimated volume (mm3)')
ax.set_title('OpenPlaque plaque-volume proxy by artery')
ax.legend(['Low attenuation', 'Noncalcified', 'Mixed/intermediate', 'Calcified'], bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plot_path = PLOT_DIR / 'best_estimate_plaque_types_by_artery.png'
plt.savefig(plot_path, dpi=150)
plt.show()

html_path = OUTPUT_DIR / 'best_estimate_plaque_types_by_artery.html'
html = f"""<!doctype html><html><head><meta charset="utf-8"><title>Best-estimate plaque types by artery</title>
<style>body{{font-family:-apple-system,BlinkMacSystemFont,Segoe UI,sans-serif;margin:32px;color:#222;}} table{{border-collapse:collapse;width:100%;font-size:13px;}} th,td{{border:1px solid #ddd;padding:6px 8px;text-align:right;}} th:first-child,td:first-child{{text-align:left;}} th{{background:#f4f4f4;}} .notice{{background:#fff3cd;border:1px solid #ffe08a;padding:12px 14px;border-radius:8px;}} .small{{font-size:12px;color:#555;}}</style></head><body>
<h1>Best-estimate plaque types by artery</h1>
<p class="notice"><b>Research use only.</b> These are OpenPlaque proxy estimates. PROMISE and CONFIRM2 thresholds were derived from validated quantitative CCTA tools using lumen and outer-vessel-wall plaque contours; the comparison columns below are orientation aids, not clinical classifications.</p>
<p>Method: classify nnU-Net plaque-core voxels by HU; add vessel-context candidate voxels outside labels 1 and 2 for low attenuation (-30 to &lt;30 HU), noncalcified (30 to &lt;130 HU), and mixed/intermediate (130 to &lt;350 HU). Calcified burden is core voxels &gt;=350 HU. Volumes use artery DICOM spacing when plausible, otherwise UCLA Series 7/reference CTA spacing, rather than mask-file or generated NIfTI spacing. <b>Total plaque volume proxy</b> equals strict nnU-Net core plus selected context candidates, so it may exceed literature TPV definitions.</p>
{estimate_df[display_cols].to_html(index=False, float_format=lambda x: f'{x:.3f}')}
<p class="small">PROMISE reference: TPV 87.2 mm3. CONFIRM2 references: TPV stages 0, &gt;0-250, &gt;250-750, &gt;750 mm3; NCPV stages 0-20, &gt;20-125, &gt;125-375, &gt;375 mm3.</p>
<p>CSV: {csv_path.name}<br>Plot: {plot_path.name}</p>
</body></html>"""
html_path.write_text(html, encoding='utf-8')
print('CSV:', csv_path)
print('HTML:', html_path)
print('Plot:', plot_path)
